In [ ]:
import numpy as np
import pandas as pd
from nltk.classify import NaiveBayesClassifier
from nltk.corpus import subjectivity
from nltk.sentiment import SentimentAnalyzer
from nltk.sentiment.util import *

#pd.set_option('display.max_colwidth', -1)
from time import time
import re
import string
import os
from pprint import pprint
import collections
import matplotlib.pyplot as plt
# %matplotlib inline

import seaborn as sns
sns.set(style="darkgrid")
sns.set(font_scale=1.3)

from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import GridSearchCV
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline, FeatureUnion
from sklearn.metrics import classification_report

from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
#from sklearn.externals import joblib
import joblib as jb

#import gensim

from nltk.corpus import stopwords
from nltk.stem import PorterStemmer
from nltk.tokenize import word_tokenize

import warnings
warnings.filterwarnings('ignore')

np.random.seed(37)

SHOW_GRAPHS = True

In [ ]:
import datetime, pytz
from time import timezone
df_vix = pd.read_csv("/Users/Farid Srouji/Documents/Research/master/Data/VIX/vix-1m.csv",delimiter=";", decimal=",",names=["Date", "Minute", "Open", "High","Low","Close","Volume"])
# tz_chicago = pytz.timezone('America/Chicago')
df_vix.index = pd.to_datetime(df_vix['Date'] + ' ' + df_vix['Minute'], utc=True)\
                      .map(lambda x: x.tz_convert('America/Chicago'))#,dtype='datetime64[ns, Chi/Kolkata]')#tzinfo=tz_chicago)
df_vix["Open"] = df_vix["Open"].astype(str).astype(float)
df_vix["High"] = df_vix["High"].astype(str).astype(float)
df_vix["Low"] = df_vix["Low"].astype(str).astype(float)
df_vix["Close"] = df_vix["Close"].astype(str).astype(float)
df_vix["Volume"] = df_vix["Volume"].astype(str).astype(float)
# df_vix = df_vix["2020-06-01":"2022-06-01"].sort_index()
df_vix.index = pd.to_datetime(df_vix.index)
df_vix["Date"]= pd.to_datetime(df_vix.index)
df_vix.rename(columns={"Open":"VIX"},inplace=True)

# df_vix = df_vix.loc[df_vix["Date"]>"2020-05-31"]

df_vix["VIX(t-1)"] = df_vix["VIX"].shift(periods=1)
df_vix["VIX(t-1)"].ffill(inplace=True)

df_vix["vix_ln_return"]=np.log(df_vix["VIX"]/df_vix["VIX(t-1)"])
df_vix["vix_ln_return"].ffill(inplace=True)

df_vix["Volume(t-1)"] = df_vix["Volume"].shift(periods=1)
df_vix["Volume(t-1)"].ffill(inplace=True)

df_vix["Volume_ln_return"]=np.log(df_vix["Volume"]/df_vix["Volume(t-1)"])
df_vix["Volume_ln_return"].ffill(inplace=True)

In [ ]:
df_vix

In [ ]:
df_sentiment = pd.read_pickle("/Users/Farid Srouji/Documents/Research/master/financialmodelingprep/data/pkl/SentimentTimeTypeText.pkl")

In [ ]:
df_sentiment

In [ ]:
# df_sentiment.drop(columns=["compound", "Negative", "Positive", "neu", "Articles", "Title", "Text", "Company"], inplace=True)
# df_sentiment.drop(columns=["Company"], inplace=True)
df_sentiment.index = df_sentiment.Time
df_sentiment.index = pd.to_datetime(df_sentiment.index)
df_sentiment.Date = df_sentiment.index

df_sentiment = df_sentiment.ffill()

df_sentiment["Sentiment(t-1)"] = df_sentiment["Sentiment"].shift(periods=1)
df_sentiment["Sentiment(t-2)"] = df_sentiment["Sentiment"].shift(periods=2)
df_sentiment["Sentiment(t-1)"] = df_sentiment["Sentiment(t-1)"].ffill()
df_sentiment["Sentiment(t-2)"] = df_sentiment["Sentiment(t-2)"].ffill()

# SENTIMENT RETURN
# df_sentiment["Sentiment_return"]=(df_sentiment["Sentiment(t-1)"]-df_sentiment["Sentiment(t-2)"])/df_sentiment["Sentiment(t-2)"]
# SENTIMENT LN RETURN
df_sentiment["Sentiment Natural Log"]=np.log(df_sentiment["Sentiment"]/df_sentiment["Sentiment(t-1)"])
df_sentiment["Sentiment Natural Log(t-1)"]=np.log(df_sentiment["Sentiment(t-1)"]/df_sentiment["Sentiment(t-2)"])

# SENTIMENT EWM
df_sentiment["Sentiment_ewm10"]=df_sentiment["Sentiment"].ewm(10).mean()
df_sentiment["Sentiment_ewm20"]=df_sentiment["Sentiment"].ewm(20).mean()

# POSITIVE LN RETURN
df_sentiment["Positive(t-1)"] = df_sentiment["Positive"].shift(periods=1)
df_sentiment["Positive(t-2)"] = df_sentiment["Positive"].shift(periods=2)
df_sentiment["Positive(t-1)"] = df_sentiment["Positive(t-1)"].ffill()
df_sentiment["Positive(t-2)"] = df_sentiment["Positive(t-2)"].ffill()
df_sentiment["Positive Natural Log"]=np.log(df_sentiment["Positive"]/df_sentiment["Positive(t-1)"])
df_sentiment["Positive Natural Log(t-1)"]=np.log(df_sentiment["Positive(t-1)"]/df_sentiment["Positive(t-2)"])

# NEGATIVE LN RETURN
df_sentiment["Negative(t-1)"] = df_sentiment["Negative"].shift(periods=1)
df_sentiment["Negative(t-2)"] = df_sentiment["Negative"].shift(periods=2)
df_sentiment["Negative(t-1)"] = df_sentiment["Negative(t-1)"].ffill()
df_sentiment["Negative(t-2)"] = df_sentiment["Negative(t-2)"].ffill()
df_sentiment["Negative Natural Log"]=np.log(df_sentiment["Negative"]/df_sentiment["Negative(t-1)"])
df_sentiment["Negative Natural Log(t-1)"]=np.log(df_sentiment["Negative(t-1)"]/df_sentiment["Negative(t-2)"])

# # NEUTRAL LN RETURN
# df_sentiment["Neutral(t-1)"] = df_sentiment["Neutral"].shift(periods=1)
# df_sentiment["Neutral(t-2)"] = df_sentiment["Neutral"].shift(periods=2)
# df_sentiment["Neutral(t-1)"] = df_sentiment["Neutral(t-1)"].ffill()
# df_sentiment["Neutral(t-2)"] = df_sentiment["Neutral(t-2)"].ffill()
# df_sentiment["Neutral Natural Log"]=np.log(df_sentiment["Neutral"]/df_sentiment["Neutral(t-1)"])
# df_sentiment["Neutral Natural Log(t-1)"]=np.log(df_sentiment["Neutral(t-1)"]/df_sentiment["Neutral(t-2)"])

# # COMPOUND LN RETURN
# df_sentiment["Compound(t-1)"] = df_sentiment["Compound"].shift(periods=1)
# df_sentiment["Compound(t-2)"] = df_sentiment["Compound"].shift(periods=2)
# df_sentiment["Compound(t-1)"] = df_sentiment["Compound(t-1)"].ffill()
# df_sentiment["Compound(t-2)"] = df_sentiment["Compound(t-2)"].ffill()
# df_sentiment["Compound Natural Log"]=np.log(df_sentiment["Compound"]/df_sentiment["Compound(t-1)"])
# df_sentiment["Compound Natural Log(t-1)"]=np.log(df_sentiment["Compound(t-1)"]/df_sentiment["Compound(t-2)"])

df_sentiment["Polarity(t-1)"] = df_sentiment["Polarity"].shift(periods=1)
df_sentiment["Subjectivity(t-1)"] = df_sentiment["Subjectivity"].shift(periods=1)

df_sentiment['WordCount'] = df_sentiment['Articles'].apply(lambda x:len(str(x).split())) #Number Of words in main text
df_sentiment["WordCount(t-1)"] = df_sentiment["WordCount"].shift(periods=1)
df_sentiment["WordCount(t-2)"] = df_sentiment["WordCount"].shift(periods=2)
df_sentiment["WordCount(t-1)"] = df_sentiment["WordCount(t-1)"].ffill()
df_sentiment["WordCount(t-2)"] = df_sentiment["WordCount(t-2)"].ffill()
df_sentiment["WordCount Natural Log"]=np.log(df_sentiment["WordCount"]/df_sentiment["WordCount(t-1)"])

In [ ]:
(df_sentiment["Positive Natural Log"]!=-np.inf).sum()

In [ ]:
(df_sentiment["Positive Natural Log"]>0 ).sum()

In [ ]:
df_sentiment["2022-04-28"]

In [ ]:
df_sentiment.index = df_sentiment.Time

In [ ]:
df_vix

In [ ]:
df = df_vix.merge(df_sentiment,how="left",right_on="Date", left_on="Date")
df = df.ffill()
df = df.sort_values(by="Date")

In [ ]:
df = df_vix.merge(df_sentiment,how="left",right_on="Date", left_on="Date")


In [ ]:
df

In [ ]:
df.loc[df["vix_ln_return"]==0]#.head(50)

In [ ]:
import datetime as dt
from datetime import date
import calendar
my_date = date.today()
calendar.day_name[my_date.weekday()]  #'Wednesday'

# Prepare data
df['Year'] = [d.year for d in df.Date]
df['Month'] = [d.strftime('%b') for d in df.Date]
df['Day of Week'] = [d.day_name() for d in df.Date]
# df['Hour'] = [d.hour() for d in df.Hour]
# df['Minute'] = [d.minute() for d in df.Minute]

years = df['Year'].unique()
df['Month-Year'] = [d.strftime('%b') + str(d.year) for d in df.Date]
df['Quarter'] = pd.PeriodIndex(df.Date, freq='Q')

mx = df.loc[df['Sentiment Natural Log'] != np.inf, 'Sentiment Natural Log'].max()
df['Sentiment Natural Log'].replace(np.inf,mx,inplace=True)
mn = df.loc[df['Sentiment Natural Log'] != -np.inf, 'Sentiment Natural Log'].min()
df['Sentiment Natural Log'].replace(-np.inf,mn,inplace=True)

mx = df.loc[df['vix_ln_return'] != np.inf, 'vix_ln_return'].max()
df['vix_ln_return'].replace(np.inf,mx,inplace=True)
mn = df.loc[df['vix_ln_return'] != -np.inf, 'vix_ln_return'].min()
df['vix_ln_return'].replace(-np.inf,mn,inplace=True)

mx = df.loc[df['Positive Natural Log'] != np.inf, 'Positive Natural Log'].max()
df['Positive Natural Log'].replace(np.inf,mx,inplace=True)
mn = df.loc[df['Positive Natural Log'] != -np.inf, 'Positive Natural Log'].min()
df['Positive Natural Log'].replace(-np.inf,mn,inplace=True)

mx = df.loc[df['Negative Natural Log'] != np.inf, 'Negative Natural Log'].max()
df['Negative Natural Log'].replace(np.inf,mx,inplace=True)
mn = df.loc[df['Negative Natural Log'] != -np.inf, 'Negative Natural Log'].min()
df['Negative Natural Log'].replace(-np.inf,mn,inplace=True)

In [ ]:
df.loc[df["vix_ln_return"]==0]#.head(50)

In [ ]:
df.index = df.Time

In [ ]:
df.sort_index(inplace=True)

In [ ]:
df.rename(columns={"vix":"VIX",
                "vix(t-1)":"VIX(t-1)",
                "vix_ln_return":"VIX Natural Log",
                "sp500":"S&P500",
                "sp500(t-1)":"S&P500(t-1)",
                "sp500(t-2)":"S&P500(t-2)",
                "sp500_ln_return":"S&P500 Natural Log",
                "Sentiment_ln_return":"Sentiment Natural Log",
                "Sentiment_ln_return(t-1)":"Sentiment Natural Log(t-1)",
                "pos":"Positive",
                "pos(t-1)":"Positive(t-1)",
                "pos(t-2)":"Positive(t-2)",
                "pos_ln_return":"Positive Natural Log",
                "pos_ln_return(t-1)":"Positive Natural Log(t-1)",
                "neg":"Negative",
                "neg(t-1)":"Negative(t-1)", 
                "neg(t-2)":"Negative(t-2)",
                "neg_ln_return":"Negative Natural Log", 
                "neg_ln_return(t-1)":"Negative Natural Log(t-1)", 
                "neu":"Neutral", 
                "neu(t-1)":"Neutral(t-1)",
                "neu(t-2)":"Neutral(t-2)",
                "neu_ln_return":"Neutral Natural Log",
                "neu_ln_return(t-1)":"Neutral Natural Log(t-1)",
                "compound":"Compound",
                "compound(t-1)":"Compound(t-1)",
                "compound(t-2)":"Compound(t-2)",
                "compound_ln_return":"Compound Natural Log",
                "compound_ln_return(t-1)":"Compound Natural Log(t-1)",
                "polarity":"Polarity",
                "polarity(t-1)":"Polarity(t-1)",
                "polarityB":"PolarityB",
                "subjectivity":"Subjectivity",
                "subjectivity(t-1)":"Subjectivity(t-1)",
                "PressReleaseCount_ln_return":"PressReleaseCount Natural Log",
                "PressReleaseCount_ln_return(t-1)":"PressReleaseCount Natural Log(t-1)",
                "WordCount_ln_return":"WordCount Natural Log",
                "WordCount_ln_return(t-1)":"WordCount Natural Log(t-1)",
                "day":"Day of Week",
                "month":"Month",
                "month-year":"Month-Year",
                "quarter":"Quarter",
                "year":"Year",
                "GI_Sentiment_ln_return":"GI_Sentiment Natural Log",
                "GI_Negative_ln_return":"GI_Negative Natural Log",
                "GI_Positive_ln_return":"GI_Positive Natural Log"}
                ,inplace=True)

In [ ]:
df.bfill(inplace=True)

In [ ]:
# df1.dropna()

In [ ]:
df.to_pickle("/Users/Farid Srouji/Documents/Research/master/financialmodelingprep/data/pkl/dfTime.pkl")